In [1]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np
import faiss
import time
import json

/home/intern/MyWork/rag/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7317.11it/s]


In [3]:
with open("500DaysofSummer.txt", "r", encoding="utf-8") as file:
    text = file.read()

In [4]:
def chunk_text(text, chunk_size=400, overlap=100):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)
    return chunks

def chunk_by_paragraph(text, max_size=400):
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    chunks = []
    current = ""
    for para in paragraphs:
        if len(current) + len(para) + 2 <= max_size:
            current = current + "\n" + para if current else para
        else:
            if current:
                chunks.append(current)
            if len(para) > max_size:
                for i in range(0, len(para), max_size - 50):
                    chunks.append(para[i:i + max_size])
            else:
                current = para
    if current:
        chunks.append(current)
    return chunks

chunks = chunk_by_paragraph(text)
print(f"Total chunks: {len(chunks)}")

Total chunks: 269


In [5]:
emb = model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)

emb = emb.astype("float32")
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

In [20]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY2")
from groq import Groq

client = Groq(api_key=api_key)
# import importlib, groq_deepeval
# importlib.reload(groq_deepeval)
# from groq_deepeval import GroqModel
# groq_model = GroqModel(api_key, model="llama-3.1-8b-instant")

In [9]:
query = "What was douchebag referring to in the movie?"
query_embedding = model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
)

query_embedding = query_embedding.reshape(1, -1).astype("float32")
retrieved_chunks = []
distances, indices = index.search(query_embedding, 10)
for idx in indices[0]:
    retrieved_chunks.append(chunks[idx])

context = "\n\n".join(retrieved_chunks)

distances, indices = index.search(query_embedding, 10)

In [ ]:
prompt = f"""
You are a question-answering assistant.

Use ONLY the provided context.

Rules:

1. Never use outside knowledge.
2. Never infer information that is not explicitly stated.
3. If the answer is missing, reply exactly:
   "No information available in the provided context."
4. After every answer, include the exact sentence(s) from the context that support your answer.
5. If no supporting sentence exists, return only:
   "No information available in the provided context."

Context:
{context}

Question:
{query}

"""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)
print("Question:" ,query)
print("Answer:" ,response.choices[0].message.content)

In [19]:
with open("test_dataset.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

In [20]:
def ask_rag(query, search_query=None, history=None):
    if search_query is None:
        search_query = query
    if history is None:
        history = []

    query_embedding = model.encode([search_query]).astype("float32")
    distances, indices = index.search(query_embedding, 20)
    candidate_chunks = [chunks[i] for i in indices[0]]

    pairs = [[search_query, chunk] for chunk in candidate_chunks]
    scores = reranker.predict(pairs)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:5]
    retrieved_chunks = [candidate_chunks[i] for i in top_indices]

    context = "

".join(retrieved_chunks)

    history_text = ""
    for turn in history[-5:]:
        history_text += "User: " + turn["user"] + "
"
        history_text += "Assistant: " + turn["assistant"] + "

"

    history_block = f"Conversation History:
{history_text}
" if history_text else ""

    prompt = f"""You are a precise question-answering assistant.

Use ONLY the context below to answer. Do not use any outside knowledge.
Be concise and direct. If the answer is not in the context, say exactly:
"No information available in the provided context."

{history_block}Retrieved Context:
{context}

Current Question:
{query}

Answer:"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    answer = response.choices[0].message.content
    return answer, retrieved_chunks


In [7]:
import json
import os

# Load previous results
if os.path.exists("rag_outputs_reranked(2).json"):
    with open("rag_outputs_reranked(2).json", "r", encoding="utf-8") as f:
        rag_output = json.load(f)
else:
    rag_output = []

print("Already completed:", len(rag_output))

Already completed: 160


In [8]:
start = len(rag_output)

print("Resuming from question", start + 1)

Resuming from question 161


In [ ]:
for item in dataset[start:]:

    question = item["question"]
    expected_answer = item["answer"]

    print(f"\nRunning: {question}")

    try:
        rag_answer, retrieved_chunks = ask_rag(question)

        rag_output.append({
            "question": question,
            "expected_answer": expected_answer,
            "rag_answer": rag_answer,
            "retrieved_chunks": retrieved_chunks
        })

        # Save immediately after each successful question
        with open("rag_outputs_reranked.json", "w", encoding="utf-8") as f:
            json.dump(rag_output, f, indent=4, ensure_ascii=False)

        print("✓ Saved", len(rag_output), "answers")

    except Exception as e:
        print("Error:", e)
        print("Progress saved. Resume later.")
        break


Running: What is the name of the boy in the story?
✓ Saved 1 answers

Running: Where is Tom from?
✓ Saved 2 answers

Running: What is the name of the girl in the story?
✓ Saved 3 answers

Running: Where is Summer from?
✓ Saved 4 answers

Running: What does Tom believe in?
✓ Saved 5 answers

Running: What is Summer's attitude towards love?
✓ Saved 6 answers

Running: What is Tom's occupation?
✓ Saved 7 answers

Running: What is Summer's job?
✓ Saved 8 answers

Running: How does Tom meet Summer?
✓ Saved 9 answers

Running: What is the date when Tom meets Summer?
✓ Saved 10 answers

Running: What is the name of Tom's half-sister?
✓ Saved 11 answers

Running: Why does Summer want to stop seeing Tom?
✓ Saved 12 answers

Running: What is the name of the song quoted by Summer in her high school yearbook?
✓ Saved 13 answers

Running: What is the name of the album by Belle & Sebastian that experiences a spike in sales in Michigan?
✓ Saved 14 answers

Running: Where does Summer work during her 

In [20]:
print(test_cases[0].__dict__)

IndexError: list index out of range

In [12]:
import json

from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    HallucinationMetric
)
from deepeval.test_case import LLMTestCase

faithfulness = FaithfulnessMetric(model=groq_model)
answer_relevancy = AnswerRelevancyMetric(model=groq_model)
hallucination = HallucinationMetric(model=groq_model)

In [18]:
import json
from deepeval.test_case import LLMTestCase

with open("rag_outputs_partial.json", "r", encoding="utf-8") as f:
    rag_outputs = json.load(f)

test_cases = []

for item in rag_outputs:

    question = item["question"]
    rag_answer = item["rag_answer"]
    expected_answer = item["expected_answer"]
    retrieved_chunks = item["retrieved_chunks"]

    tc = LLMTestCase(
        input=question,
        actual_output=rag_answer,
        expected_output=expected_answer,
        retrieval_context=retrieved_chunks,
        context=retrieved_chunks
    )

    test_cases.append(tc)

print(len(test_cases))

180


In [ ]:
import os

EVAL_RESULTS_FILE = "eval_results.json"
small_test = test_cases[:20]

if os.path.exists(EVAL_RESULTS_FILE):
    with open(EVAL_RESULTS_FILE, "r") as f:
        eval_results = json.load(f)
else:
    eval_results = []

done_questions = {r["question"] for r in eval_results}
print(f"Already evaluated: {len(eval_results)}, Remaining: {len(small_test) - len(eval_results)}")

for i, tc in enumerate(small_test):
    if tc.input in done_questions:
        print(f"[{i+1}/{len(small_test)}] Skipping (already done): {tc.input}")
        continue

    print(f"
[{i+1}/{len(small_test)}] Q: {tc.input}")
    try:
        faithfulness.measure(tc)
        answer_relevancy.measure(tc)
        hallucination.measure(tc)

        result = {
            "question": tc.input,
            "faithfulness": faithfulness.score,
            "answer_relevancy": answer_relevancy.score,
            "hallucination": hallucination.score,
        }
        eval_results.append(result)

        with open(EVAL_RESULTS_FILE, "w") as f:
            json.dump(eval_results, f, indent=4)

        print(f"  Faithfulness:     {faithfulness.score:.2f} ({'PASS' if faithfulness.is_successful() else 'FAIL'})")
        print(f"  Answer Relevancy: {answer_relevancy.score:.2f} ({'PASS' if answer_relevancy.is_successful() else 'FAIL'})")
        print(f"  Hallucination:    {hallucination.score:.2f} ({'PASS' if hallucination.is_successful() else 'FAIL'})")
    except Exception as e:
        print(f"  ERROR: {e}")
        print("  Skipping and continuing...")

    time.sleep(10)

print("
" + "="*40)
print("FINAL SUMMARY")
print("="*40)
for metric in ["faithfulness", "answer_relevancy", "hallucination"]:
    vals = [r[metric] for r in eval_results]
    if vals:
        print(f"{metric:20s}: avg={sum(vals)/len(vals):.2f}, min={min(vals):.2f}, max={max(vals):.2f}")

In [28]:
import os, json, time
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
_api_key = os.getenv("GROQ_API_KEY2")
judge_client = Groq(api_key=_api_key)

EVAL_FILE = "llm_judge_results_improvise(2).json"
SOURCE_FILE = "rag_outputs_reranked(2).json"

if os.path.exists(EVAL_FILE):
    with open(EVAL_FILE) as f:
        judge_results = json.load(f)
else:
    judge_results = []

done_questions = {r["question"] for r in judge_results}

with open(SOURCE_FILE) as f:
    rag_outputs = json.load(f)

print(f"Total: {len(rag_outputs)} | Already done: {len(judge_results)} | Remaining: {len(rag_outputs) - len(judge_results)}")

for item in rag_outputs:
    question = item["question"]
    if question in done_questions:
        continue

    context = "".join(item["retrieved_chunks"])
    rag_answer = item["rag_answer"]
    expected_answer = item["expected_answer"]

    prompt = f"""
You are evaluating a RAG system.

Question:
{question}

Retrieved Context:
{context}

Expected Answer:
{expected_answer}

RAG Answer:
{rag_answer}

Evaluate using semantic meaning, not exact wording.

Scoring rules:

- Ignore capitalization and minor wording differences.
- Partial names count (Tom = Tom Hansen, Summer = Summer Finn).
- Short but correct answers are acceptable.
- Score based ONLY on the retrieved context.

Faithfulness:
- 1.0 = answer is fully supported by the retrieved context.
- 0.5–0.9 = mostly supported.
- 0.0 = unsupported or contradicts the context.
- If the context lacks the requested information and the RAG correctly replies "No information available in the provided context.", give Faithfulness = 1.0.

Relevancy:
- High if the answer addresses the question.
- Low only if it is off-topic or refuses despite sufficient context.

Correctness:
- Compare with the expected answer by meaning, not wording.
- Partial answers deserve partial credit.
- If the RAG correctly refuses because the information is absent from the retrieved context, Correctness = 0.0.

Return ONLY valid JSON:

{{
  "faithfulness": 0.0,
  "relevancy": 0.0,
  "correctness": 0.0,
  "reason": "One short sentence. If the RAG gives no answer, write: No answer provided by the RAG system."
}}
"""

    try:
        response = judge_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200,
            temperature=0.0
        )
        raw = response.choices[0].message.content.strip()
        start, end = raw.index("{"), raw.rindex("}") + 1
        scores = json.loads(raw[start:end])

        judge_results.append({
            "question": question,
            "rag_answer": rag_answer,
            "expected_answer": expected_answer,
            "faithfulness": scores.get("faithfulness"),
            "relevancy": scores.get("relevancy"),
            "correctness": scores.get("correctness"),
            "reason": scores.get("reason")
        })
        done_questions.add(question)

        with open(EVAL_FILE, "w") as f:
            json.dump(judge_results, f, indent=4, ensure_ascii=False)

        print(f"[{len(judge_results)}/{len(rag_outputs)}] {question[:55]} | F:{scores.get('faithfulness')} R:{scores.get('relevancy')} C:{scores.get('correctness')}")
        time.sleep(10)

    except Exception as e:
        print(f"Error: {e}")
        print("Progress saved. Re-run to resume.")
        break

if len(judge_results) == len(rag_outputs):
    valid = [r for r in judge_results if r["faithfulness"] is not None]
    print(f"=== FINAL SCORES ({len(valid)} questions) ===")
    print(f"Avg Faithfulness: {sum(r['faithfulness'] for r in valid)/len(valid):.2f}")
    print(f"Avg Relevancy:    {sum(r['relevancy'] for r in valid)/len(valid):.2f}")
    print(f"Avg Correctness:  {sum(r['correctness'] for r in valid)/len(valid):.2f}")

Total: 160 | Already done: 154 | Remaining: 6


In [ ]:
import os, json, time
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
_api_key = os.getenv("GROQ_API_KEY")
judge_client = Groq(api_key=_api_key)

EVAL_FILE = "llm_judge_results.json"
SOURCE_FILE = "rag_outputs_reranked(2).json"

if os.path.exists(EVAL_FILE):
    with open(EVAL_FILE) as f:
        saved = json.load(f)
    judge_results = [r for r in saved if "question" in r]
else:
    judge_results = []

done_questions = {r["question"] for r in judge_results}

with open(SOURCE_FILE) as f:
    rag_outputs = json.load(f)

print(f"Total: {len(rag_outputs)} | Already done: {len(judge_results)} | Remaining: {len(rag_outputs) - len(judge_results)}")

for item in rag_outputs:
    question = item["question"]
    if question in done_questions:
        continue

    context = "\n\n".join(item["retrieved_chunks"])
    rag_answer = item["rag_answer"]
    expected_answer = item["expected_answer"]

    prompt = f"""You are a lenient but fair evaluation judge for a RAG system.

Question: {question}
Retrieved Context: {context}
RAG Answer: {rag_answer}
Expected Answer: {expected_answer}

Score each metric from 0.0 to 1.0 using these guidelines:

faithfulness: Is the RAG answer supported by the retrieved context?
- 1.0 = answer is clearly supported by the context (even if context uses different casing or phrasing)
- 0.5 = answer is partially supported
- 0.0 = answer contradicts or is completely absent from context

relevancy: Does the RAG answer address the question asked?
- 1.0 = directly answers the question
- 0.5 = partially answers the question
- 0.0 = completely off-topic or refuses to answer

correctness: Does the RAG answer match the expected answer in meaning?
- 1.0 = same meaning, partial names/answers count (e.g. "Tom" matches "Tom Hansen", "doctor" matches "Paul is a doctor")
- 0.5 = partially correct
- 0.0 = completely wrong or no answer given

Reply with only valid JSON:
{{"faithfulness": 0.0, "relevancy": 0.0, "correctness": 0.0, "reason": "brief reason"}}"""

    try:
        response = judge_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200,
            temperature=0.0
        )
        raw = response.choices[0].message.content.strip()
        start, end = raw.index("{"), raw.rindex("}") + 1
        scores = json.loads(raw[start:end])

        judge_results.append({
            "question": question,
            "rag_answer": rag_answer,
            "expected_answer": expected_answer,
            "faithfulness": scores.get("faithfulness"),
            "relevancy": scores.get("relevancy"),
            "correctness": scores.get("correctness"),
            "reason": scores.get("reason")
        })
        done_questions.add(question)

        with open(EVAL_FILE, "w") as f:
            json.dump(judge_results, f, indent=4, ensure_ascii=False)

        print(f"[{len(judge_results)}/{len(rag_outputs)}] {question[:55]} | F:{scores.get('faithfulness')} R:{scores.get('relevancy')} C:{scores.get('correctness')}")
        time.sleep(10)

    except Exception as e:
        print(f"Error: {e}")
        print("Progress saved. Re-run to resume.")
        break

valid = [r for r in judge_results if r["correctness"] is not None]
if valid:
    total = len(valid)
    correct_count = sum(1 for r in valid if r["correctness"] >= 0.5)
    accuracy = correct_count / total * 100
    avg_f = sum(r["faithfulness"] for r in valid if r["faithfulness"] is not None) / total
    avg_r = sum(r["relevancy"] for r in valid if r["relevancy"] is not None) / total
    avg_c = sum(r["correctness"] for r in valid) / total

    summary = {
        "_summary": {
            "total_questions": total,
            "accuracy": round(accuracy, 1),
            "correct_count": correct_count,
            "avg_faithfulness": round(avg_f, 2),
            "avg_relevancy": round(avg_r, 2),
            "avg_correctness": round(avg_c, 2)
        }
    }

    with open(EVAL_FILE, "w") as f:
        json.dump([summary] + judge_results, f, indent=4, ensure_ascii=False)

    print(f"\n=== SUMMARY ({total} questions) ===")
    print(f"Accuracy (correctness >= 0.5): {correct_count}/{total} = {accuracy:.1f}%")
    print(f"Avg Faithfulness: {avg_f:.2f}")
    print(f"Avg Relevancy:    {avg_r:.2f}")
    print(f"Avg Correctness:  {avg_c:.2f}")


In [ ]:
def rewrite_query(query, history):
    if len(history) == 0:
        return query

    history_text = ""
    for turn in history[-5:]:
        history_text += "User: " + turn["user"] + "
"
        history_text += "Assistant: " + turn["assistant"] + "

"

    prompt = f"""You are given a conversation.

Rewrite ONLY the latest user question into a standalone question.

Conversation:
{history_text}
Latest Question:
{query}

Standalone Question:"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()


def chat(query, history):
    rewritten = rewrite_query(query, history)
    answer, chunks = ask_rag(query=query, search_query=rewritten, history=history)
    history.append({"user": query, "assistant": answer})
    return answer, history


history = []
print("Chatbot ready. Type quit to exit.
")

while True:
    query = input("You: ").strip()
    if query.lower() in ("quit", "exit", "q"):
        print("Bye!")
        break
    if not query:
        continue
    answer, history = chat(query, history)
    print(f"
Assistant: {answer}
")
